1
A cnn uses a convolutional layer that helps with feature extraction. These layers apply filters on the input, helping the model detect patterns and features. In a 2d convolutional layer (the one we are using in this problem set and project), a kernel slides over the input data (in our case an image) and combines the values it is currently on using elementwise multiplication into a single output. This is repeated as it is slid over the input. Feed forward networks do not use convolutional layers. 

CNNs are useful for computer vision problems because they can learn spatial relationships within images, they help with dimensionality reduction, and the overlapping that occurs when the step the kernel takes is smaller than its size helps it capture fine details and patterns that span multiple pixels. 

An autoencoder is a type of neural network that is trained to compress and then decompress data. This process is called encoding and decoding. 

By using non linear activation functions (relu), the autoencoder can learn complex and non linear mappings between the input and latent space. 

If the autoencoder uses linear activation functions, the autoencoder can be used for linear dimensionality reduction. When an autoencoder that uses linear activation functions uses MSE (mean squared error) as the reconstruction loss, it minimizes the squared difference between input and reconstruction. This is almost identical to the reconstruction error in PCA. 

In [30]:
import tensorflow as tf
from tensorflow.keras import layers, Model, models
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import os
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [ ]:
# Load and preprocess images
def load_and_preprocess_images(folder_path, target_size=(64, 64)):  # Change target size to 64x64
    images = []
    for filename in os.listdir(folder_path):
        img_path = os.path.join(folder_path, filename)
        img = load_img(img_path, target_size=target_size)
        img_array = img_to_array(img) / 255.0  # Normalize to [0, 1]
        images.append(img_array)
    return np.array(images)

# Load Lightning and Rainbow class images
lightning_images = load_and_preprocess_images('dataset1/dew')
rainbow_images = load_and_preprocess_images('dataset1/sandstorm')

# Combine the datasets
all_images = np.concatenate([lightning_images, rainbow_images])

# Split the dataset into training and validation sets
X_train, X_val = train_test_split(all_images, test_size=0.2, random_state=42)

# Define the Autoencoder model
def build_autoencoder(input_shape, latent_dim):
    # Encoder
    encoder_input = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_input)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 64x64 -> 32x32
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 32x32 -> 16x16
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 16x16 -> 8x8
    x = layers.Flatten()(x)
    encoder_output = layers.Dense(latent_dim)(x)

    encoder = models.Model(encoder_input, encoder_output, name='encoder')

    # Decoder
    decoder_input = layers.Input(shape=(latent_dim,))
    x = layers.Dense(8 * 8 * 128, activation='relu')(decoder_input)  # Adjust for 8x8 feature maps
    x = layers.Reshape((8, 8, 128))(x)
    x = layers.Conv2DTranspose(128, (3, 3), strides=2, activation='relu', padding='same')(x)  # 8x8 -> 16x16
    x = layers.Conv2DTranspose(64, (3, 3), strides=2, activation='relu', padding='same')(x)   # 16x16 -> 32x32
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(x)   # 32x32 -> 64x64
    decoder_output = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)        # Output 64x64x3

    decoder = models.Model(decoder_input, decoder_output, name='decoder')

    # Autoencoder
    autoencoder_input = layers.Input(shape=input_shape)
    encoded = encoder(autoencoder_input)
    decoded = decoder(encoded)
    autoencoder = models.Model(autoencoder_input, decoded, name='autoencoder')

    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder, encoder, decoder

# Latent dimensions to test
latent_dims = [2, 4, 8, 16, 32]
input_shape = X_train.shape[1:]  # This will now be (64, 64, 3)
reconstructed_images_dict = {}  # To store reconstructed images for each latent dimension

# Train and evaluate Autoencoder for each latent dimension
for latent_dim in latent_dims:
    print(f"Training Autoencoder with latent dimension: {latent_dim}")
    
    # Build the Autoencoder
    autoencoder, encoder, decoder = build_autoencoder(input_shape, latent_dim)

    # Train the Autoencoder
    history = autoencoder.fit(X_train, X_train,
                              epochs=50,
                              batch_size=32,
                              validation_data=(X_val, X_val),
                              verbose=0)  # Suppress training output for clarity

    # Reconstruct images from the validation set
    reconstructed_images = autoencoder.predict(X_val)
    reconstructed_images_dict[latent_dim] = reconstructed_images  # Save reconstructed images

# Function to plot reconstructed images
def plot_reconstructed_images(original_images, reconstructed_images, latent_dim, n=5):
    plt.figure(figsize=(20, 4))
    for i in range(n):
        # Display original
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(original_images[i])
        plt.title("Original")
        plt.axis('off')

        # Display reconstruction
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(reconstructed_images[i])
        plt.title(f"Reconstructed (Latent Dim = {latent_dim})")
        plt.axis('off')
    plt.show()

# Plot reconstructed images for each latent dimension
for latent_dim in latent_dims:
    print(f"Reconstructed Images for Latent Dimension = {latent_dim}")
    plot_reconstructed_images(X_val, reconstructed_images_dict[latent_dim], latent_dim)

Training Autoencoder with latent dimension: 2
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step
Training Autoencoder with latent dimension: 4
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step
Training Autoencoder with latent dimension: 8
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step
Training Autoencoder with latent dimension: 16
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
Training Autoencoder with latent dimension: 32
